# ✏️ Notebook de Ejercicios
## Adquisición de Datos + Ciclo de Vida del Machine Learning

Completá los `TODO` de cada celda. La mayoría de los ejercicios tienen una celda de **auto-chequeo** con `assert` al final: si no marca error, ¡vas bien!

> 📎 Si te trabás, revisá el notebook **`Practica_Adquisicion_y_Ciclo_de_Vida_ML.ipynb`**, que tiene los mismos temas resueltos y explicados paso a paso.

### 📂 Datasets que usa este notebook

Todos estos archivos deben estar en la carpeta `datasets/` (junto a este notebook):

| Archivo | Usado en | Descripción |
|---|---|---|
| `productos.csv` | Ejercicio 1 | Catálogo de productos (2000 filas) |
| `encuesta.csv` | Ejercicio 2 | Encuesta con separador `;` y encoding `latin-1` |
| `config.json` | Ejercicio 2 (bonus) | Configuración de ejemplo en JSON |
| `inscripciones.csv` | Ejercicios 3, 4 y 9 | Inscripciones a cursos, con nulos/duplicados/inconsistencias |
| `clientes.csv` | Ejercicio 5 | Dataset chico para practicar Train/Test sin Data Leakage |

**Ejercicios**

| # | Tema |
|---|---|
| 1 | Formatos tabulares: CSV, Excel, Parquet |
| 2 | Lectura robusta de archivos (encoding y separador) |
| 3 | EDA y control de calidad de datos |
| 4 | Limpieza de datos |
| 5 | División Train/Test sin Data Leakage |
| 6 | Consumo de una API REST con `requests` |
| 7 | Paginación de resultados |
| 8 | Backoff exponencial ante Rate Limits |
| 9 | 🌟 Desafío integrador: pipeline completo |


In [ ]:
# Ejecutá esta celda primero: importa todo lo necesario para el resto del notebook
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

DATASETS = Path("datasets")
assert DATASETS.exists(), "No se encontró la carpeta 'datasets/'. Asegurate de descomprimirla junto al notebook."

print("Entorno listo ✅")
print("Archivos disponibles en datasets/:", [p.name for p in sorted(DATASETS.iterdir())])


---
## Ejercicio 1 — Formatos tabulares: CSV, Excel y Parquet

Vas a trabajar con `productos.csv`, un catálogo con 2000 filas. Tu tarea:

1. Cargar `productos.csv` en un DataFrame llamado `productos`.
2. Crear una carpeta `salidas/` (si no existe) para los archivos que vas a generar.
3. Guardar `productos` en los otros 2 formatos: `salidas/productos.xlsx` y `salidas/productos.parquet`.
4. Guardar en un diccionario `tamanos` el tamaño en KB de cada uno de los 3 archivos (claves: `"csv"`, `"xlsx"`, `"parquet"`).

**Recordatorio de la teoría:** Parquet comprime mejor los datos repetitivos (como columnas de texto con pocos valores distintos) porque almacena la información *columna por columna*, no fila por fila como el CSV.


In [ ]:
# TODO 1: cargar productos.csv en un DataFrame
productos = None

# TODO 2: crear la carpeta de salida
salida_dir = None  # <-- reemplazá con Path("salidas") y creá la carpeta con mkdir(exist_ok=True)

# TODO 3: guardar 'productos' en los otros 2 formatos dentro de salida_dir
# ruta_xlsx = ...
# ruta_parquet = ...


# TODO 4: calcular el tamaño en KB de cada uno de los 3 archivos y guardarlo en el diccionario `tamanos`
#         (el CSV original está en DATASETS / "productos.csv")
tamanos = {
    "csv": None,
    "xlsx": None,
    "parquet": None,
}

tamanos


In [ ]:
# ✅ Auto-chequeo — no modifiques esta celda
assert productos is not None and len(productos) == 2000, "No se cargó bien productos.csv"
assert salida_dir is not None and salida_dir.exists(), "La carpeta salida_dir no existe"
assert all(v is not None for v in tamanos.values()), "Todavía faltan tamaños por calcular"
assert tamanos["parquet"] < tamanos["csv"], "Se esperaba que Parquet pese menos que CSV (por la compresión)"
print("Ejercicio 1 OK ✅")


**Pregunta para pensar:** ¿por qué Parquet ocupa menos espacio que CSV para los mismos datos? Respondé en la celda de abajo (markdown).

_Tu respuesta:_ 

---
## Ejercicio 2 — Lectura robusta de archivos

`encuesta.csv` viene con separador `;` y encoding `latin-1` (simulando un archivo exportado desde Excel en configuración regional de Argentina). La lectura de abajo está **rota a propósito**. Arreglala.

**Pista:** antes de arreglarla, abrí el archivo en modo binario (`open(ruta, "rb")`) y mirá los primeros bytes con `.readline()`, como hicimos en el notebook de práctica — eso te va a mostrar pistas sobre el separador y el encoding.


In [ ]:
ruta_encuesta = DATASETS / "encuesta.csv"

# TODO: esta lectura está rota (separador y encoding incorrectos). Arreglala.
df_encuesta = pd.read_csv(ruta_encuesta)
df_encuesta


In [ ]:
# ✅ Auto-chequeo
assert list(df_encuesta.columns) == ["nombre", "satisfaccion", "comentario"], "Las columnas no se separaron bien: revisá el parámetro sep"
assert df_encuesta["nombre"].iloc[0] == "Andrés Gómez", "Los acentos no se ven bien: revisá el parámetro encoding"
print("Ejercicio 2 OK ✅")


**Bonus:** completá la función `leer_json_config` para que abra `datasets/config.json` usando el bloque `with` y devuelva el diccionario. Recordá: usar `with` garantiza que el archivo se cierre bien incluso si ocurre un error durante la lectura.

In [ ]:
def leer_json_config(ruta):
    # TODO: abrí el archivo con `with`, cargalo con json.load y devolvé el diccionario
    pass


resultado = leer_json_config(DATASETS / "config.json")
resultado


In [ ]:
# ✅ Auto-chequeo
assert resultado == {"entorno": "desarrollo", "max_reintentos": 3, "timeout": 15}, "El diccionario devuelto no coincide con el contenido de config.json"
print("Bonus OK ✅")


---
## Ejercicio 3 — EDA y control de calidad de datos

`inscripciones.csv` es un dataset de inscripciones a un curso, con problemas típicos de calidad. Explorálo **antes** de limpiarlo — nunca limpies a ciegas.


In [ ]:
inscripciones = pd.read_csv(DATASETS / "inscripciones.csv")
inscripciones


In [ ]:
# TODO 1: mostrá la cantidad de valores nulos por columna
nulos_por_columna = None

# TODO 2: mostrá cuántas filas están duplicadas (considerando todas las columnas)
cantidad_duplicados = None

# TODO 3: usando .describe(), guardá en `resumen_horas` la media de 'horas_dedicadas' (como número, no como Series)
resumen_horas = None

print(nulos_por_columna)
print("Duplicados:", cantidad_duplicados)
print("Media de horas:", resumen_horas)


In [ ]:
# ✅ Auto-chequeo
assert nulos_por_columna is not None and nulos_por_columna["curso"] == 1, "Revisá el conteo de nulos"
assert cantidad_duplicados == 1, "Se esperaba encontrar 1 fila duplicada (id=6)"
assert isinstance(resumen_horas, (int, float)), "resumen_horas debe ser un número, no una Series"
print("Ejercicio 3 OK ✅")


Completá la función `chequear_esquema`, que dado un DataFrame y una lista de columnas esperadas, devuelva un diccionario con:
- `"columnas_faltantes"`: lista de columnas esperadas que no están en el DataFrame.
- `"pct_nulos"`: diccionario `{columna: porcentaje_de_nulos}` para cada columna del DataFrame.

(Esto simula un chequeo de **Data Contract**, como vimos en el notebook de práctica y en la clase de Adquisición de Datos: un acuerdo formal sobre qué forma deben tener los datos.)

In [ ]:
def chequear_esquema(df: pd.DataFrame, columnas_esperadas: list) -> dict:
    # TODO: completar
    reporte = {"columnas_faltantes": [], "pct_nulos": {}}
    return reporte


reporte = chequear_esquema(inscripciones, columnas_esperadas=["id", "curso", "horas_dedicadas", "aprobado", "fecha_inscripcion"])
reporte


In [ ]:
# ✅ Auto-chequeo
assert reporte["columnas_faltantes"] == ["fecha_inscripcion"], "No detectaste bien la columna faltante"
assert round(reporte["pct_nulos"]["curso"], 2) == round(1/9, 2), "El % de nulos de 'curso' no es correcto"
print("Ejercicio 3 (bonus) OK ✅")


---
## Ejercicio 4 — Limpieza de datos

Usando el mismo `inscripciones`, generá `inscripciones_limpio` aplicando (en este orden):

1. Eliminar filas duplicadas.
2. Normalizar la columna `curso` para que `"python"`/`"Python"` sean iguales, `"sql"`/`"SQL"` sean iguales, etc. (usá `.str.strip().str.title()` o similar).
3. Completar los nulos de `curso` con el valor `"Sin especificar"`.
4. Completar los nulos de `horas_dedicadas` con la **mediana** de la columna (¿por qué mediana y no promedio? Repasalo en el notebook de práctica, sección 6).
5. Eliminar las filas donde `aprobado` sea nulo (no podemos inventar si aprobó o no: sería un dato demasiado sensible para completar arbitrariamente).


In [ ]:
inscripciones_limpio = inscripciones.copy()

# TODO 1: eliminar duplicados


# TODO 2: normalizar texto de 'curso'


# TODO 3: completar nulos de 'curso'


# TODO 4: completar nulos de 'horas_dedicadas' con la mediana


# TODO 5: eliminar filas con 'aprobado' nulo


inscripciones_limpio


In [ ]:
# ✅ Auto-chequeo
assert inscripciones_limpio.duplicated().sum() == 0, "Todavía hay duplicados"
assert inscripciones_limpio["curso"].isnull().sum() == 0, "Todavía hay nulos en 'curso'"
assert inscripciones_limpio["horas_dedicadas"].isnull().sum() == 0, "Todavía hay nulos en 'horas_dedicadas'"
assert inscripciones_limpio["aprobado"].isnull().sum() == 0, "Todavía hay nulos en 'aprobado'"
assert inscripciones_limpio["curso"].str.lower().nunique() <= 4, "Revisá la normalización de 'curso': no deberían quedar duplicados por mayúsculas"
print("Ejercicio 4 OK ✅")


---
## Ejercicio 5 — División Train/Test sin Data Leakage

`clientes.csv` tiene un valor faltante en `ingreso_mensual`. Dividí correctamente en train/test (70/30, `random_state=7`) y luego imputá los nulos usando **solo estadísticas del set de entrenamiento** (sin Data Leakage). Repasá en el notebook de práctica (sección 7) por qué el orden "dividir → limpiar" es crítico.


In [ ]:
clientes = pd.read_csv(DATASETS / "clientes.csv")

X = clientes[["ingreso_mensual", "edad"]]
y = clientes["compra"]

# TODO 1: dividir en train/test (70/30, random_state=7)
X_train, X_test, y_train, y_test = None, None, None, None

# TODO 2: calcular la media de 'ingreso_mensual' SOLO con X_train
media_ingreso_train = None

# TODO 3: imputar los nulos de 'ingreso_mensual' en X_train y X_test usando esa media
# (no recalcules la media con X_test; usá la de train para ambos)


print("Media usada para imputar:", media_ingreso_train)
print("Nulos en train:", X_train["ingreso_mensual"].isnull().sum())
print("Nulos en test:", X_test["ingreso_mensual"].isnull().sum())


In [ ]:
# ✅ Auto-chequeo
assert X_train is not None and len(X_train) == 7, "El tamaño de train no es el esperado (70% de 10 = 7)"
assert len(X_test) == 3, "El tamaño de test no es el esperado (30% de 10 = 3)"
assert X_train["ingreso_mensual"].isnull().sum() == 0, "Todavía hay nulos en train"
assert X_test["ingreso_mensual"].isnull().sum() == 0, "Todavía hay nulos en test"
print("Ejercicio 5 OK ✅")


**Pregunta para pensar:** si en vez de dividir primero, hubiéramos calculado la media de `ingreso_mensual` con **todo** el dataset (antes de dividir) y luego dividido, ¿qué problema se produce? Respondé abajo.

_Tu respuesta:_ 

---
## Ejercicio 6 — Consumo de una API REST con `requests`

Vamos a usar la API pública de GitHub (no requiere autenticación para uso básico). Completá la función para que:

1. Haga un `GET` a `https://api.github.com/repos/{owner}/{repo}`.
2. Si `status_code == 200`, devuelva una tupla `(nombre_completo, cantidad_de_estrellas, lenguaje)`.
3. Si no, devuelva `(None, None, None)`.

Recordá el significado de los códigos de estado HTTP que vimos en el notebook de práctica (200, 403, 404, 429...).


In [ ]:
def info_repo(owner: str, repo: str):
    url = f"https://api.github.com/repos/{owner}/{repo}"
    # TODO: completar
    respuesta = None
    if respuesta is not None and respuesta.status_code == 200:
        datos = respuesta.json()
        return None  # TODO: devolver la tupla (nombre_completo, estrellas, lenguaje)
    return None, None, None


nombre, estrellas, lenguaje = info_repo("psf", "requests")
print(nombre, estrellas, lenguaje)


In [ ]:
# ✅ Auto-chequeo (si ves un AssertionError de conexión, puede ser el rate-limit compartido: probá de nuevo en unos minutos)
if nombre is not None:
    assert nombre == "psf/requests"
    assert isinstance(estrellas, int)
    print("Ejercicio 6 OK ✅")
else:
    print("⚠️ No se pudo consultar la API ahora mismo (revisá tu conexión o el rate limit). Revisá igual que la función esté bien escrita.")


---
## Ejercicio 7 — Paginación de resultados

Completá `traer_todos_los_repos`, que debe traer los repositorios públicos de una organización de GitHub usando paginación por `page`/`per_page`, hasta un máximo de páginas indicado, deteniéndose si una página viene vacía (repasá el patrón en la sección 9 del notebook de práctica).

Endpoint: `https://api.github.com/orgs/{org}/repos`


In [ ]:
def traer_todos_los_repos(org: str, por_pagina: int = 5, max_paginas: int = 3):
    resultados = []
    url = f"https://api.github.com/orgs/{org}/repos"

    for pagina in range(1, max_paginas + 1):
        # TODO 1: armar el dict de params con 'per_page' y 'page'
        params = None

        # TODO 2: hacer el GET
        r = None

        # TODO 3: si status_code no es 200, cortar el bucle (break)


        # TODO 4: parsear el JSON, si viene vacío cortar el bucle, si no, extenderlo en `resultados`


    return resultados


repos_encontrados = traer_todos_los_repos("octokit", por_pagina=5, max_paginas=2)
print(f"Repos encontrados: {len(repos_encontrados)}")


In [ ]:
# ✅ Auto-chequeo
if repos_encontrados:
    assert isinstance(repos_encontrados, list)
    assert all("name" in r for r in repos_encontrados)
    print("Ejercicio 7 OK ✅")
else:
    print("⚠️ La lista vino vacía: puede ser rate-limit de la API en este entorno. Revisá que tu función esté bien escrita igualmente.")


---
## Ejercicio 8 — Backoff exponencial ante Rate Limits

Completá la función `get_con_reintentos`, que:

1. Intenta hacer `requests.get(url, timeout=10)`.
2. Si el `status_code` es `200`, devuelve la respuesta inmediatamente.
3. Si el `status_code` es `429`, `403` o `500`, espera `delay` segundos, **duplica** `delay` (backoff exponencial), y reintenta — hasta `max_reintentos` veces.
4. Si se agotan los reintentos, devuelve la última respuesta igual (no lanza una excepción).

Recordá por qué el backoff se duplica en cada intento (en vez de esperar siempre lo mismo): repasalo en la sección 9 del notebook de práctica.


In [ ]:
def get_con_reintentos(url, max_reintentos=4, delay_inicial=1):
    delay = delay_inicial
    r = None
    for intento in range(1, max_reintentos + 1):
        # TODO: completar el cuerpo del bucle según las reglas de arriba
        pass
    return r


resp = get_con_reintentos("https://api.github.com/repos/pandas-dev/pandas")
print("Status final:", resp.status_code if resp is not None else None)


In [ ]:
# ✅ Auto-chequeo
assert resp is not None, "La función no debe devolver None: siempre debe devolver la última respuesta obtenida"
assert resp.status_code in (200, 403, 429, 500), "El status code no es uno de los esperados"
print("Ejercicio 8 OK ✅ (revisá también que hayas usado time.sleep con backoff exponencial)")


---
## Ejercicio 9 — 🌟 Desafío integrador: pipeline completo

Armá una función `pipeline_inscripciones(ruta_entrada, ruta_salida)` que combine todo lo visto, siguiendo la misma estructura del pipeline reproducible del notebook de práctica (sección 10):

1. **Ingesta**: leer el CSV desde `ruta_entrada`.
2. **Validar esquema**: debe tener las columnas `id`, `curso`, `horas_dedicadas`, `aprobado` (usá `chequear_esquema` del Ejercicio 3, o escribí la validación de nuevo con un `assert`).
3. **Limpiar**: aplicar los mismos pasos del Ejercicio 4 (duplicados, normalizar texto, nulos).
4. **Guardar**: escribir el resultado como Parquet en `ruta_salida`, con nombre `inscripciones_procesado.parquet`.
5. **Devolver** el DataFrame limpio.


In [ ]:
def pipeline_inscripciones(ruta_entrada: Path, ruta_salida: Path) -> pd.DataFrame:
    columnas_esperadas = ["id", "curso", "horas_dedicadas", "aprobado"]

    # TODO 1: leer el CSV desde ruta_entrada
    df_crudo = None

    # TODO 2: validar que todas las columnas esperadas estén presentes (assert)


    # TODO 3: limpiar el dataset (duplicados, texto normalizado, nulos)
    df = df_crudo.copy()


    # TODO 4: guardar como parquet en ruta_salida / "inscripciones_procesado.parquet"


    return df


salida_dir_9 = Path("salidas")
salida_dir_9.mkdir(exist_ok=True)

df_final = pipeline_inscripciones(DATASETS / "inscripciones.csv", salida_dir_9)
df_final


In [ ]:
# ✅ Auto-chequeo final
assert (salida_dir_9 / "inscripciones_procesado.parquet").exists(), "No se guardó el archivo parquet"
assert df_final.duplicated().sum() == 0, "El resultado final todavía tiene duplicados"
assert df_final.isnull().sum().sum() == 0, "El resultado final todavía tiene nulos"
print("🎉 ¡Desafío integrador completo! Pipeline reproducible funcionando de punta a punta.")


---
## ¿Cómo seguir?

- Repasá los ejercicios que te costaron más comparando con el notebook de práctica.
- Probá los ejercicios 6-8 con otra API pública que te interese (por ejemplo, alguna que uses en tu proyecto final).
- Como desafío extra, agregále al pipeline del Ejercicio 9 un log que registre cuántas filas se eliminaron en cada paso de limpieza.
